# ResearchFlow Skills - Interactive Demo

Ta notebook demonstrira uporabo ResearchFlow Skills sistema za avtomatizirano pisanje scoping review člankov.

## 📚 Vsebina

1. [Setup](#1-setup)
2. [Nalaganje Skills](#2-skills)
3. [Templates](#3-templates)
4. [Schema Validacija](#4-schemas)
5. [IRR Calculator](#5-irr)
6. [PRISMA Generator](#6-prisma)
7. [Full Workflow](#7-workflow)

## 1. Setup {#1-setup}

Najprej naložimo potrebne module.

In [ ]:
import sys
from pathlib import Path

# Dodaj projekt v path
project_root = Path().resolve().parent if 'notebooks' in str(Path().resolve()) else Path().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")

In [ ]:
# Osnovni imports
from agents import (
    get_skill, 
    get_system_prompt, 
    list_all_skills,
    SkillLoader,
    SkillMetadata
)
from agents.skills import (
    list_templates,
    load_template
)
from agents.skills.templates.schemas import (
    validate_data_charting,
    create_empty_data_charting,
    list_schemas
)

print("✅ Imports successful!")

## 2. Nalaganje Skills {#2-skills}

ResearchFlow ima 20 skills razporejenih v 3 cluster-je.

In [ ]:
# Seznam vseh skills
skills = list_all_skills()
print(f"📚 Total skills: {len(skills)}\n")

# Grupiraj po cluster-ju
clusters = {}
for skill in skills:
    parts = skill.split('/')
    cluster = parts[0]
    if cluster not in clusters:
        clusters[cluster] = []
    clusters[cluster].append(skill)

for cluster, cluster_skills in clusters.items():
    print(f"\n📁 {cluster}:")
    for s in cluster_skills:
        name = s.split('/')[-1] if '/' in s else '(entry point)'
        print(f"    - {name}")

In [ ]:
# Naloži specifičen skill
researcher_skill = get_skill("research-cluster/researcher")

print(f"📖 Skill: {researcher_skill.name}")
print(f"\n📝 Description:")
print(researcher_skill.description[:300] + "...")
print(f"\n🛠️ Tools: {researcher_skill.tools}")
print(f"🤖 Model: {researcher_skill.model}")
print(f"📂 Cluster: {researcher_skill.cluster}")

In [ ]:
# Pridobi system prompt za LLM
system_prompt = get_system_prompt("researcher")

print("💬 System Prompt za Researcher Agent:")
print("-" * 50)
print(system_prompt)

In [ ]:
# Preglej sekcije v skill.md
print(f"📑 Parsed sections in researcher skill:")
for section_name, content in researcher_skill.sections.items():
    preview = content[:100].replace('\n', ' ') + "..." if len(content) > 100 else content.replace('\n', ' ')
    print(f"   - {section_name}: {preview}")

## 3. Templates {#3-templates}

Standardne predloge za scoping review workflow.

In [ ]:
# Seznam templates
templates = list_templates()

print(f"📄 Available templates ({len(templates)}):")
for t in templates:
    print(f"   - {t}")

In [ ]:
# Naloži data charting form
charting_form = load_template("data-charting-form")

print("📝 Data Charting Form (preview):")
print("-" * 50)
# Prikaži prvih 50 vrstic
lines = charting_form.split('\n')[:50]
for line in lines:
    print(line)

In [ ]:
# Naloži screening form
screening_form = load_template("screening-form")

print("🔍 Screening Form (preview):")
print("-" * 50)
lines = screening_form.split('\n')[:40]
for line in lines:
    print(line)

## 4. Schema Validacija {#4-schemas}

JSON Schema validacija za data charting.

In [ ]:
# Seznam shem
schemas = list_schemas()
print(f"📋 Available schemas: {schemas}")

In [ ]:
# Ustvari prazno strukturo
empty_form = create_empty_data_charting()

print("📝 Empty data charting structure:")
import json
print(json.dumps(empty_form, indent=2))

In [ ]:
# Primer validnih podatkov
valid_extraction = {
    "study_id": "Smith2024a",
    "reviewer": "Research Assistant",
    "extraction_date": "2024-04-16",
    "bibliographic": {
        "authors": "Smith, J.; Doe, A.; Johnson, B.",
        "year": 2024,
        "title": "AI-Powered Hiring: Effects on Employee Wellbeing",
        "journal": "Journal of Applied Psychology",
        "doi": "10.1037/apl0001234"
    },
    "methodology": {
        "study_design": {
            "type": "quantitative",
            "subtype": "cross-sectional survey",
            "data_collection": ["survey"]
        },
        "sample": {
            "size": 342,
            "response_rate": 67.5
        }
    },
    "pcc": {
        "population": {
            "primary": "Job applicants exposed to AI screening"
        },
        "concept": {
            "ai_technologies": ["resume_screening", "video_interviews"],
            "hr_functions": ["recruitment", "selection"]
        },
        "context": {
            "geographic_scope": "United States"
        }
    }
}

# Validiraj
errors = validate_data_charting(valid_extraction)

if errors:
    print("❌ Validation errors:")
    for err in errors:
        print(f"   - {err}")
else:
    print("✅ Validation passed! Data is valid.")

In [ ]:
# Primer neveljavnih podatkov
invalid_extraction = {
    "study_id": "Smith2024a",
    # Manjka reviewer
    "extraction_date": "2024-04-16",
    "bibliographic": {
        "authors": "Smith, J.",
        "year": 2030,  # Leto izven veljavnega razpona
        "title": "Test Study"
    },
    "methodology": {},
    "pcc": {}
}

errors = validate_data_charting(invalid_extraction)

print("❌ Expected validation errors:")
for err in errors:
    print(f"   - {err}")

## 5. IRR Calculator {#5-irr}

Inter-Rater Reliability z Cohen's Kappa.

In [ ]:
try:
    from agents.skills.templates.scripts import calculate_kappa, IRRCalculator
    IRR_AVAILABLE = True
    print("✅ IRR Calculator available")
except ImportError:
    IRR_AVAILABLE = False
    print("⚠️ IRR Calculator requires numpy and scipy:")
    print("   pip install numpy scipy")

In [ ]:
if IRR_AVAILABLE:
    # Primer: Screening decisions od dveh reviewer-jev
    reviewer1_decisions = [
        "include", "include", "exclude", "include", "exclude",
        "include", "exclude", "include", "include", "exclude",
        "include", "include", "exclude", "exclude", "include",
        "exclude", "include", "include", "exclude", "include"
    ]

    reviewer2_decisions = [
        "include", "exclude", "exclude", "include", "exclude",
        "include", "include", "include", "include", "exclude",
        "include", "include", "exclude", "exclude", "exclude",
        "exclude", "include", "include", "exclude", "include"
    ]

    result = calculate_kappa(reviewer1_decisions, reviewer2_decisions)

    print("📊 Inter-Rater Reliability Results")
    print("=" * 40)
    print(f"Cohen's Kappa: κ = {result['kappa']:.3f}")
    print(f"Percent Agreement: {result['percent_agreement']:.1f}%")
    print(f"Interpretation: {result['interpretation']}")
    print(f"\nTotal items: {result['n_items']}")
    print(f"Agreements: {result['n_agreements']}")
    print(f"Disagreements: {result['n_disagreements']}")
else:
    print("⏭️ Skipping - IRR Calculator not available")

In [ ]:
if IRR_AVAILABLE:
    # Uporaba IRRCalculator razreda za več detajlov
    calc = IRRCalculator()
    
    # Dodaj posamezne ratinge
    studies = [
        ("S001", "include", "include"),
        ("S002", "include", "exclude"),
        ("S003", "exclude", "exclude"),
        ("S004", "include", "include"),
        ("S005", "uncertain", "include"),
    ]
    
    for study_id, r1, r2 in studies:
        calc.add_rating(study_id, r1, r2)
    
    result = calc.calculate()
    
    print("📊 Detailed IRR Analysis")
    print(f"Kappa: {result['kappa']:.3f} ({result['interpretation']})")
    
    # Confusion matrix
    print("\n📋 Confusion Matrix:")
    matrix = calc.get_confusion_matrix()
    print(matrix)
    
    # Disagreements
    print("\n⚠️ Disagreements:")
    for d in result['disagreements']:
        print(f"   {d['item_id']}: R1={d['rater1']}, R2={d['rater2']}")
else:
    print("⏭️ Skipping - IRR Calculator not available")

## 6. PRISMA Generator {#6-prisma}

Generiranje PRISMA flow diagramov.

In [ ]:
try:
    from agents.skills.templates.scripts import generate_prisma_diagram, PRISMAGenerator
    PRISMA_AVAILABLE = True
    print("✅ PRISMA Generator available")
except ImportError:
    PRISMA_AVAILABLE = False
    print("⚠️ PRISMA Generator not available")

In [ ]:
if PRISMA_AVAILABLE:
    # Definiraj števila za PRISMA
    prisma_counts = {
        "identified": 1547,
        "duplicates_removed": 312,
        "screened": 1235,
        "excluded_screening": 987,
        "sought_retrieval": 248,
        "not_retrieved": 23,
        "assessed_eligibility": 225,
        "excluded_eligibility": 108,
        "included": 117
    }

    # ASCII diagram
    ascii_diagram = generate_prisma_diagram(prisma_counts, format="ascii")
    print(ascii_diagram)
else:
    print("⏭️ Skipping - PRISMA Generator not available")

In [ ]:
if PRISMA_AVAILABLE:
    # Mermaid format za dokumentacijo
    mermaid_diagram = generate_prisma_diagram(prisma_counts, format="mermaid")
    print("Mermaid diagram (for Markdown rendering):")
    print("-" * 50)
    print(mermaid_diagram)
else:
    print("⏭️ Skipping")

In [ ]:
if PRISMA_AVAILABLE:
    # JSON format za nadaljnjo obdelavo
    json_data = generate_prisma_diagram(prisma_counts, format="json")
    import json
    print("JSON structure:")
    print(json_data)
else:
    print("⏭️ Skipping")

## 7. Full Workflow Example {#7-workflow}

Primer celotnega workflow-a za scoping review.

In [ ]:
# 1. Naloži skills za research cluster
print("STEP 1: Load Research Cluster Skills")
print("=" * 50)

research_skills = [
    "research-cluster/researcher",
    "research-cluster/literature-scout",
    "research-cluster/data-extractor"
]

for skill_path in research_skills:
    skill = get_skill(skill_path)
    print(f"\n🔬 {skill.name}")
    print(f"   {skill.description.split(chr(10))[0]}")

In [ ]:
# 2. Pripravi search documentation
print("STEP 2: Search Strategy Documentation")
print("=" * 50)

search_template = load_template("search-documentation")
# Prikaži strukturo
print(search_template[:1500])

In [ ]:
# 3. Screening z IRR
print("STEP 3: Screening with IRR Monitoring")
print("=" * 50)

screening_template = load_template("screening-form")
print("Screening form structure:")
print(screening_template[:800])

In [ ]:
# 4. Data extraction z validacijo
print("STEP 4: Data Extraction & Validation")
print("=" * 50)

# Primer ekstrakcije
extracted_study = {
    "study_id": "Chen2023b",
    "reviewer": "Lead Reviewer",
    "extraction_date": "2024-04-16",
    "bibliographic": {
        "authors": "Chen, L.; Wang, M.",
        "year": 2023,
        "title": "Algorithmic Management and Employee Stress"
    },
    "methodology": {
        "study_design": {
            "type": "mixed",
            "subtype": "sequential explanatory",
            "data_collection": ["survey", "interview"]
        }
    },
    "pcc": {
        "population": {"primary": "Gig workers"},
        "concept": {
            "ai_technologies": ["monitoring", "analytics"],
            "hr_functions": ["performance"]
        },
        "context": {"geographic_scope": "China"}
    }
}

errors = validate_data_charting(extracted_study)
if errors:
    print("❌ Errors found:")
    for e in errors:
        print(f"   {e}")
else:
    print("✅ Extraction validated successfully")
    print(json.dumps(extracted_study, indent=2))

In [ ]:
# 5. PRISMA checklist
print("STEP 5: PRISMA-ScR Compliance Check")
print("=" * 50)

prisma_checklist = load_template("prisma-scr-checklist")
print(prisma_checklist[:1500])

In [ ]:
# 6. Quality cluster skills
print("STEP 6: Quality Cluster Evaluation")
print("=" * 50)

quality_skills = [
    "quality-cluster/multi-evaluator",
    "quality-cluster/fact-checker",
    "quality-cluster/methodology-validator"
]

for skill_path in quality_skills:
    skill = get_skill(skill_path)
    print(f"\n✅ {skill.name}")
    print(f"   {skill.description.split(chr(10))[0]}")

## 📊 Summary

Ta notebook demonstrira ključne komponente ResearchFlow Skills sistema.

In [ ]:
print("\n" + "=" * 60)
print("RESEARCHFLOW SKILLS SUMMARY")
print("=" * 60)
print(f"\n📚 Total Skills: {len(list_all_skills())}")
print(f"📄 Templates: {len(list_templates())}")
print(f"📋 Schemas: {len(list_schemas())}")
print(f"\n🔬 Research Cluster: {len([s for s in list_all_skills() if 'research' in s])} skills")
print(f"✍️ Writing Cluster: {len([s for s in list_all_skills() if 'writing' in s])} skills")
print(f"✅ Quality Cluster: {len([s for s in list_all_skills() if 'quality' in s])} skills")
print("\n" + "=" * 60)